In [0]:
from pyspark.sql import functions as F, types as T

### Brands

In [0]:
df_bronze = spark.table("ecommerce.bronze.brz_brands")
df_bronze.show(5)

+----------+-----------+-------------+--------------------+--------------------+
|brand_code| brand_name|Category_code|        _source_file|         ingested_at|
+----------+-----------+-------------+--------------------+--------------------+
|      ACME|   AcmeTech|           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
|      NOVW|  NovaWave |           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
|      ZNTH|     Zenith|           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
|      BYTM|    ByteMax|           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
|      ECOT|    EcoTone|           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
+----------+-----------+-------------+--------------------+--------------------+
only showing top 5 rows


In [0]:
df_silver = df_bronze.withColumn("brand_name",F.trim(F.col("brand_name")))
df_silver.show(5)


+----------+----------+-------------+--------------------+--------------------+
|brand_code|brand_name|Category_code|        _source_file|         ingested_at|
+----------+----------+-------------+--------------------+--------------------+
|      ACME|  AcmeTech|           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
|      NOVW|  NovaWave|           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
|      ZNTH|    Zenith|           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
|      BYTM|   ByteMax|           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
|      ECOT|   EcoTone|           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
+----------+----------+-------------+--------------------+--------------------+
only showing top 5 rows


In [0]:
df_silver = df_silver.withColumn("brand_code",F.regexp_replace(F.col("brand_code"),r'[^A-Za-z0-9]',''))
df_silver.show(10)

+----------+----------+-------------+--------------------+--------------------+
|brand_code|brand_name|Category_code|        _source_file|         ingested_at|
+----------+----------+-------------+--------------------+--------------------+
|      ACME|  AcmeTech|           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
|      NOVW|  NovaWave|           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
|      ZNTH|    Zenith|           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
|      BYTM|   ByteMax|           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
|      ECOT|   EcoTone|           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
|      SKYL|   SkyLink|           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
|      VOLT|  VoltEdge|           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
|      PHTX|  Photonix|           CE|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
|      URTL|UrbanTrail|          APP|dbfs:/Volumes/eco...|2026-09-03 16:45:...|
|      COTC|CottonClub|          APP|dbf

In [0]:
df_silver.select("Category_code").distinct().show()

+-------------+
|Category_code|
+-------------+
|           CE|
|          APP|
|          HNK|
|          BPC|
|        BOOKS|
|          BKS|
|      GROCERY|
|         GRCY|
|          TOY|
|         TOYS|
|          SPT|
+-------------+



In [0]:
anomalies = {
    "GROCERY" : "GRCY",
    "BOOKS" : "BKS",
    "TOYS" : "TOY"
}
df_silver = df_silver.replace(anomalies,subset="Category_code")
df_silver.select("Category_code").distinct().show()


+-------------+
|Category_code|
+-------------+
|           CE|
|          APP|
|          HNK|
|          BPC|
|          BKS|
|         GRCY|
|          TOY|
|          SPT|
+-------------+



In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeschema",True)\
    .saveAsTable("ecommerce.silver.silver_brands")

### Category

In [0]:
df_bronze = spark.table("ecommerce.bronze.brz_category")
df_bronze.show(10)

+-------------+--------------------+--------------------+--------------------+
|category_code|       category_name|        _ingested_at|        _source_file|
+-------------+--------------------+--------------------+--------------------+
|           ce|         Electronics|2026-09-03 16:57:...|dbfs:/Volumes/eco...|
|          app|             Apparel|2026-09-03 16:57:...|dbfs:/Volumes/eco...|
|          hnk|      Home & Kitchen|2026-09-03 16:57:...|dbfs:/Volumes/eco...|
|          bpc|Beauty & Personal...|2026-09-03 16:57:...|dbfs:/Volumes/eco...|
|          bks|               Books|2026-09-03 16:57:...|dbfs:/Volumes/eco...|
|         grcy|             Grocery|2026-09-03 16:57:...|dbfs:/Volumes/eco...|
|          toy|        Toys & Games|2026-09-03 16:57:...|dbfs:/Volumes/eco...|
|          spt|   Sports & Outdoors|2026-09-03 16:57:...|dbfs:/Volumes/eco...|
|          app|             Apparel|2026-09-03 16:57:...|dbfs:/Volumes/eco...|
|         grcy|             Grocery|2026-09-03 16:57

In [0]:
df_duplicates = df_bronze.groupBy("category_code").count().filter(F.col("count")>1)
df_duplicates.show()

+-------------+-----+
|category_code|count|
+-------------+-----+
|          app|    2|
|         grcy|    2|
+-------------+-----+



In [0]:
df_silver = df_bronze.dropDuplicates(["category_code"])
df_silver.show()

+-------------+--------------------+--------------------+--------------------+
|category_code|       category_name|        _ingested_at|        _source_file|
+-------------+--------------------+--------------------+--------------------+
|           ce|         Electronics|2026-09-03 16:57:...|dbfs:/Volumes/eco...|
|          app|             Apparel|2026-09-03 16:57:...|dbfs:/Volumes/eco...|
|          hnk|      Home & Kitchen|2026-09-03 16:57:...|dbfs:/Volumes/eco...|
|          bpc|Beauty & Personal...|2026-09-03 16:57:...|dbfs:/Volumes/eco...|
|          bks|               Books|2026-09-03 16:57:...|dbfs:/Volumes/eco...|
|         grcy|             Grocery|2026-09-03 16:57:...|dbfs:/Volumes/eco...|
|          toy|        Toys & Games|2026-09-03 16:57:...|dbfs:/Volumes/eco...|
|          spt|   Sports & Outdoors|2026-09-03 16:57:...|dbfs:/Volumes/eco...|
+-------------+--------------------+--------------------+--------------------+



In [0]:
df_silver  = df_silver.withColumn("category_code",F.upper(F.col("category_code")))
display(df_silver)

category_code,category_name,_ingested_at,_source_file
CE,Electronics,2026-09-03T16:57:22.497Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
APP,Apparel,2026-09-03T16:57:22.497Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
HNK,Home & Kitchen,2026-09-03T16:57:22.497Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
BPC,Beauty & Personal Care,2026-09-03T16:57:22.497Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
BKS,Books,2026-09-03T16:57:22.497Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
GRCY,Grocery,2026-09-03T16:57:22.497Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
TOY,Toys & Games,2026-09-03T16:57:22.497Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
SPT,Sports & Outdoors,2026-09-03T16:57:22.497Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv


In [0]:
df_silver.write.mode("overwrite").format("delta").option("mergeschema",True).saveAsTable("ecommerce.silver.slv_category")

### Products

In [0]:
df_bronze = spark.table("ecommerce.bronze.brz_products")
row_count,column_count = df_bronze.count(),len(df_bronze.columns)
print(f"Row Count : {row_count}")
print(f"Column Count : {column_count}")


Row Count : 50000
Column Count : 14


In [0]:
display(df_bronze.limit(5))

product_id,sku,category_code,brand_code,color,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,file_name,ingest_timestamp
2000000000015,STCR-HNK-00001,hnk,stcr,White,One-Size,Coton,305g,"22,2",17.1,6.3,0,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-09-04T04:36:53.190Z
2000000000022,HMNS-HNK-00002,hnk,hmns,Silver,One-Size,Steel,682g,"18,2",12.3,3.7,1,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-09-04T04:36:53.190Z
2000000000039,NOVW-CE-00003,ce,novw,Purple,One-Size,Wood,243g,"18,2",13.9,4.2,0,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-09-04T04:36:53.190Z
2000000000046,URTL-APP-00004,app,urtl,Silver,S,Ruber,225g,"17,6",4.6,5.8,50,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-09-04T04:36:53.190Z
2000000000053,GGRN-GRC-00005,grcy,ggrn,Silver,One-Size,Ruber,455g,"27,2",15.8,7.4,-4,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-09-04T04:36:53.190Z


In [0]:
from pyspark.sql import functions as F,types as T
df_silver = df_bronze.withColumn("weight_grams",F.regexp_replace(F.col("weight_grams"),'g','')\
    .cast(T.IntegerType()))
df_silver.select("weight_grams").show(5,truncate=False)

+------------+
|weight_grams|
+------------+
|305         |
|682         |
|243         |
|225         |
|455         |
+------------+
only showing top 5 rows


In [0]:
df_silver.select("length_cm").show(5)

+---------+
|length_cm|
+---------+
|     22,2|
|     18,2|
|     18,2|
|     17,6|
|     27,2|
+---------+
only showing top 5 rows


In [0]:
df_silver = df_silver.withColumn("length_cm",F.regexp_replace(F.col("length_cm"),',','.').cast(T.FloatType()))
df_silver.select("length_cm").show(5,truncate=False)

+---------+
|length_cm|
+---------+
|22.2     |
|18.2     |
|18.2     |
|17.6     |
|27.2     |
+---------+
only showing top 5 rows


category code and brand code are in lower case make it into upeer case

In [0]:
df_silver.select("brand_code","category_code").show(3)

+----------+-------------+
|brand_code|category_code|
+----------+-------------+
|      stcr|          hnk|
|      hmns|          hnk|
|      novw|           ce|
+----------+-------------+
only showing top 3 rows


In [0]:
df_silver = df_silver.withColumn("brand_code",F.upper(F.col("brand_code")))\
    .withColumn("category_code",F.upper(F.col("category_code")))

df_silver.select("brand_code","category_code").show(3)


+----------+-------------+
|brand_code|category_code|
+----------+-------------+
|      STCR|          HNK|
|      HMNS|          HNK|
|      NOVW|           CE|
+----------+-------------+
only showing top 3 rows


In [0]:
df_silver.select("material").distinct().show()

+---------+
| material|
+---------+
|    Coton|
|    Steel|
|     Wood|
|    Ruber|
|  Plastic|
|Polyester|
|    Glass|
|  Alumium|
|    Paper|
|  Leather|
+---------+



In [0]:
df_silver.withColumn("material",F.when(F.col("material") =="Coton","Cotton")
                     .when(F.col("material") == "Alumium","Aluminum")
                     .when(F.col("material" )== "Ruber","Rubber")
                     .otherwise(F.col("material"))).select("material").distinct().show()


+---------+
| material|
+---------+
|   Cotton|
|    Steel|
|     Wood|
|   Rubber|
|  Plastic|
|Polyester|
|    Glass|
| Aluminum|
|    Paper|
|  Leather|
+---------+



Negative values in rating count

In [0]:
df_silver.filter(F.col("rating_count")<0).select("rating_count").show()

+------------+
|rating_count|
+------------+
|          -4|
|          -2|
|          -2|
|          -1|
|         -14|
|          -1|
|          -1|
|          -6|
|         -38|
|          -2|
|          -2|
|          -3|
|          -1|
|          -6|
|         -29|
|          -5|
|          -2|
|          -5|
|          -1|
|          -5|
+------------+
only showing top 20 rows


In [0]:
# rating count should be posotive
df_silver = df_silver.withColumn("rating_count",
            F.when(F.col("rating_count").isNotNull(),F.abs(F.col("rating_count")))\
            .otherwise(F.lit(0))) # if null replace with 0

In [0]:
# Final cleaning data 

df_silver.select("brand_code","category_code","color","height_cm","weight_grams","length_cm","rating_count").show(5,truncate = False)
                 

+----------+-------------+------+---------+------------+---------+------------+
|brand_code|category_code|color |height_cm|weight_grams|length_cm|rating_count|
+----------+-------------+------+---------+------------+---------+------------+
|STCR      |HNK          |White |6.3      |305         |22.2     |0           |
|HMNS      |HNK          |Silver|3.7      |682         |18.2     |1           |
|NOVW      |CE           |Purple|4.2      |243         |18.2     |0           |
|URTL      |APP          |Silver|5.8      |225         |17.6     |50          |
|GGRN      |GRCY         |Silver|7.4      |455         |27.2     |4           |
+----------+-------------+------+---------+------------+---------+------------+
only showing top 5 rows


In [0]:
df_silver.write.format("delta").mode("overwrite").option("mergeschema",True).saveAsTable("ecommerce.silver.slv_products")

### Customers

In [0]:
# read the raw data from bronze ecommerce.bronze.brz_customers
df_bronze = spark.read.table("ecommerce.bronze.brz_customers")

row_count , column_count = df_bronze.count(),len(df_bronze.columns)

print(f"row count : {row_count}")
print(f"column ccount : {column_count}")

df_bronze.show(5)

row count : 300000
column ccount : 7
+----------------+--------------+------------+---------+-----+--------------------+--------------------+
|     customer_id|         phone|country_code|  country|state|           file_name|    ingest_timestamp|
+----------------+--------------+------------+---------+-----+--------------------+--------------------+
|CUST000000000001|917280033536.0|          IN|    India|   MH|dbfs:/Volumes/eco...|2026-09-04 04:52:...|
|CUST000000000002|619489725433.0|          AU|Australia|  VIC|dbfs:/Volumes/eco...|2026-09-04 04:52:...|
|CUST000000000003|919390066524.0|          IN|    India|   TN|dbfs:/Volumes/eco...|2026-09-04 04:52:...|
|CUST000000000004|917073741793.0|          IN|    India|   TN|dbfs:/Volumes/eco...|2026-09-04 04:52:...|
|CUST000000000005|618478772532.0|          AU|Australia|   WA|dbfs:/Volumes/eco...|2026-09-04 04:52:...|
+----------------+--------------+------------+---------+-----+--------------------+--------------------+
only showing top 5

### Handle null values in customer_id column 

In [0]:
from pyspark.sql import functions as F, types as T
null_count = df_bronze.filter(F.col("customer_id").isNull()).count()
null_count

300

In [0]:
df_silver = df_bronze.dropna(subset = ["customer_id"])
row_count = df_silver.count()
print(f"row count after dropping null values : {row_count}")

row count after dropping null values : 299700


In [0]:
null_count = df_silver.filter(F.col("phone").isNull()).count()
null_count

29964

In [0]:
df_silver.filter(F.col("phone").isNull()).show(2)

+----------------+-----+------------+-------+-----+--------------------+--------------------+
|     customer_id|phone|country_code|country|state|           file_name|    ingest_timestamp|
+----------------+-----+------------+-------+-----+--------------------+--------------------+
|CUST000000000007| NULL|          IN|  India|   MH|dbfs:/Volumes/eco...|2026-09-04 04:52:...|
|CUST000000000010| NULL|          IN|  India|   RJ|dbfs:/Volumes/eco...|2026-09-04 04:52:...|
+----------------+-----+------------+-------+-----+--------------------+--------------------+
only showing top 2 rows


In [0]:
### Fill null values with not available 
df_silver = df_silver.fillna("not available",subset = ["phone"])

## check if null still avilable 

df_silver.filter(F.col("phone").isNull()).show()


+-----------+-----+------------+-------+-----+---------+----------------+
|customer_id|phone|country_code|country|state|file_name|ingest_timestamp|
+-----------+-----+------------+-------+-----+---------+----------------+
+-----------+-----+------------+-------+-----+---------+----------------+



In [0]:
df_silver.write.format("delta").mode("overwrite").option("mergeschema",True).saveAsTable("ecommerce.silver.slv_customers")

### Calendar/date

In [0]:
df_bronze = spark.read.table("ecommerce.bronze.brz_calendar")
row_count , column_count = df_bronze.count(), len(df_bronze.columns)
print(f"row count : {row_count}")
print(f"column count : {column_count}")
df_bronze.show(5)


row count : 95
column count : 7
+----------+----+--------+-------+------------+--------------------+--------------------+
|      date|year|day_name|quarter|week_of_year|        _ingested_at|        _source_file|
+----------+----+--------+-------+------------+--------------------+--------------------+
|01-08-2025|2025|  friday|      3|         -31|2026-09-04 04:57:...|dbfs:/Volumes/eco...|
|02-08-2025|2025|SATURDAY|      3|         -31|2026-09-04 04:57:...|dbfs:/Volumes/eco...|
|03-08-2025|2025|  SUNDAY|      3|         -31|2026-09-04 04:57:...|dbfs:/Volumes/eco...|
|04-08-2025|2025|  MONDAY|      3|         -32|2026-09-04 04:57:...|dbfs:/Volumes/eco...|
|05-08-2025|2025| TUESDAY|      3|         -32|2026-09-04 04:57:...|dbfs:/Volumes/eco...|
+----------+----+--------+-------+------------+--------------------+--------------------+
only showing top 5 rows


In [0]:
df_bronze.printSchema()

root
 |-- date: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



In [0]:
df_bronze.select("date").show(5)

+----------+
|      date|
+----------+
|01-08-2025|
|02-08-2025|
|03-08-2025|
|04-08-2025|
|05-08-2025|
+----------+
only showing top 5 rows


In [0]:
## convert date column to date type
from pyspark.sql import functions as F, types as T
df_silver  = df_bronze.withColumn("date",F.to_date(F.col("date"),"dd-MM-yyyy"))

In [0]:
df_silver.printSchema()

root
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



In [0]:
duplicates = df_silver.groupBy("date").count().filter(F.col("count") > 1).show()


+----------+-----+
|      date|count|
+----------+-----+
|2025-08-29|    2|
|2025-09-25|    2|
|2025-10-13|    2|
+----------+-----+



In [0]:
# remove duplicate rows 
df_silver = df_silver.dropDuplicates(["date"])
df_silver.groupBy("date").count().filter(F.col("count") > 1).show()
print(f"Rows after removing duplicates : {df_silver.count()}")


+----+-----+
|date|count|
+----+-----+
+----+-----+

Rows after removing duplicates : 92


In [0]:
# normalize day name 

df_silver = df_silver.withColumn("day_name",F.initcap(F.col("day_name")))
df_silver.select("day_name").distinct().show()


+---------+
| day_name|
+---------+
|   Friday|
| Saturday|
|   Sunday|
|   Monday|
|  Tuesday|
|Wednesday|
| Thursday|
+---------+



In [0]:
# convert negative to positive week of year
df_silver = df_silver.withColumn("week_of_year",F.abs(F.col("week_of_year")))
df_silver.select("week_of_year").distinct().show()


+------------+
|week_of_year|
+------------+
|          31|
|          32|
|          33|
|          34|
|          35|
|          36|
|          37|
|          38|
|          39|
|          40|
|          41|
|          42|
|          43|
|          44|
+------------+



### Enhance Quarter and Week of year column

In [0]:
df_silver.select("quarter","week_of_year").show(5)

+-------+------------+
|quarter|week_of_year|
+-------+------------+
|      3|          31|
|      3|          31|
|      3|          31|
|      3|          32|
|      3|          32|
+-------+------------+
only showing top 5 rows


In [0]:
from pyspark.sql import functions as F

# Quarter in format Q3-2025
df_silver = df_silver.withColumn(
    "quarter",
    F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))
)

# Week of year in format Week31-2025
df_silver = df_silver.withColumn(
    "week_of_year",
    F.concat(F.lit("Week"), F.col("week_of_year"), F.lit("-"), F.col("year"))
)




In [0]:
df_silver.select("quarter","week_of_year").show(5)

+-------+------------+
|quarter|week_of_year|
+-------+------------+
|Q3-2025| Week31-2025|
|Q3-2025| Week31-2025|
|Q3-2025| Week31-2025|
|Q3-2025| Week32-2025|
|Q3-2025| Week32-2025|
+-------+------------+
only showing top 5 rows


In [0]:
# Rename a column
df_silver = df_silver.withColumnRenamed("week_of_year","week")
df_silver.show(2)

+----------+----+--------+-------+-----------+--------------------+--------------------+
|      date|year|day_name|quarter|       week|        _ingested_at|        _source_file|
+----------+----+--------+-------+-----------+--------------------+--------------------+
|2025-08-01|2025|  Friday|Q3-2025|Week31-2025|2026-09-04 04:57:...|dbfs:/Volumes/eco...|
|2025-08-02|2025|Saturday|Q3-2025|Week31-2025|2026-09-04 04:57:...|dbfs:/Volumes/eco...|
+----------+----+--------+-------+-----------+--------------------+--------------------+
only showing top 2 rows


In [0]:
df_silver.write.format("delta").mode("overwrite").option("mergeschema",True)\
.saveAsTable("ecommerce.silver.slv_calendar")